# Cleaning the Airparif measurements (2018-2026)

**Goal:** turn the raw CSV files downloaded from the Airparif website into clean tables, ready for the MySQL database, the exploratory analysis and the model.

**Input:** `data/raw/airparif_measurements/<year>_<POLLUTANT>.csv`, for example `2024_PM25.csv` or `2024_NO2.csv`.

**Outputs** (in `data/processed/`):

| File | Content |
|---|---|
| `stations.csv` | one row per station (code, name, type) |
| `pollutants.csv` | one row per pollutant (code, name, unit) |
| `hourly_measurements.csv` | one row per hour × station × pollutant |
| `daily_measurements.csv` | reliable daily means (18-hours-out-of-24 rule) |

**Plan:**
1. Understand the raw format
2. Read one file: headers and data
3. Go from "wide" to "long" format
4. Apply the same method to every file
5. Convert dates and values
6. Fix negative values
7. Remove duplicates
8. Build the dimension tables (stations, pollutants)
9. Check the quality (coverage rate)
10. Compute the daily means
11. Final checks and saving

In [1]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt

# Folders (the notebook is in notebooks/, the data in data/)
ROOT = Path("..")
RAW_FOLDER = ROOT / "data" / "raw" / "airparif_measurements"   # one sub-folder per source
PROCESSED_FOLDER = ROOT / "data" / "processed"
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

# Cleaning parameters, grouped here so that they are easy to justify and change
HEADER_ROWS = 6             # number of header rows in the Airparif exports
MIN_HOURS_PER_DAY = 18      # a daily mean requires at least 75 % of the 24 hours
TIMEZONE = "Europe/Paris"

files = sorted(f for f in RAW_FOLDER.rglob("*.csv")
               if re.fullmatch(r"\d{4}[ _-][A-Z0-9]+\.csv", f.name))   # "2018_PM25.csv" or "2018 PM25.csv"
print(len(files), "measurement files found:")
print([f.name for f in files])

49 measurement files found:
['2018 CO.csv', '2018 NO2.csv', '2018 NOX.csv', '2018 O3.csv', '2018 PM10.csv', '2018 PM25.csv', '2019 CO.csv', '2019 NO2.csv', '2019 NOX.csv', '2019 O3.csv', '2019 PM10.csv', '2019 PM25.csv', '2020 CO.csv', '2020 NO2.csv', '2020 NOX.csv', '2020 O3.csv', '2020 PM10.csv', '2020 PM25.csv', '2021 CO.csv', '2021 NOX.csv', '2021 O3.csv', '2021 PM10.csv', '2021 PM25.csv', '2022 CO.csv', '2022 NO2.csv', '2022 NOX.csv', '2022 O3.csv', '2022 PM10.csv', '2022 PM25.csv', '2023 CO.csv', '2023 NO2.csv', '2023 NOX.csv', '2023 O3.csv', '2023 PM25.csv', '2024 CO.csv', '2024 NO2.csv', '2024 NOX.csv', '2024 O3.csv', '2024 PM25.csv', '2025 CO.csv', '2025 NO2.csv', '2025 NOX.csv', '2025 O3.csv', '2025 PM25.csv', '2026 CO.csv', '2026 NO2.csv', '2026 NOX.csv', '2026 O3.csv', '2026 PM25.csv']


## 1. Understand the raw format

Before writing any code, we look at the file **as it is**, line by line, without pandas.

In [2]:
# A PM2.5 file as an example (the name can contain "_" or a space)
example = [f for f in files if "PM25" in f.name][0]
print("Example file:", example.name)
with open(example, encoding="utf-8") as f:
    for number in range(1, 10):
        print(number, "|", f.readline().strip()[:110])

Example file: 2018 PM25.csv
1 | ,RN6:PM25,RUR_N:PM25,RUR-SE:PM25,GON:PM25,RAMBO:PM25,BP_EST:PM25,AUT:PM25,BOB:PM25,GEN:PM25,VITRY:PM25,A1:PM25
2 | ,Route Nationale 6 - MELUN,Zone rurale Nord - Saint-Martin-du-Tertre,Zone rurale Sud-Est - Forêt de Fontainebl
3 | ,RN6,RUR_N,RUR-SE,GON,RAMBO,BP_EST,AUT,BOB,GEN,VITRY,A1,PA04C,RUR_S
4 | ,"PM 2,5 particules","PM 2,5 particules","PM 2,5 particules","PM 2,5 particules","PM 2,5 particules","PM 2,5 p
5 | ,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25,PM25
6 | ,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3,microg/m3
7 | 2018-01-01 01:00:00Z,7.1,7.1,7.9,6.4,3.0,8.9,,6.1,6.3,10.7,9.3,9.2,10.0
8 | 2018-01-01 02:00:00Z,6.5,7.7,6.9,2.8,5.4,8.2,,5.8,4.6,8.2,15.5,5.9,7.7
9 | 2018-01-01 03:00:00Z,8.2,5.5,3.7,2.6,6.0,6.1,,4.0,2.4,8.0,7.7,4.8,4.0


**What we observe:**
- **6 header rows**: `code:pollutant`, station name, station code, pollutant name, pollutant code, unit;
- then **one row per hour**, with the timestamp in **UTC** (the final `Z`);
- **one column per station**: this is a **"wide"** format.

This format is easy to read by eye, but not for a database: adding a station adds a column. We want a **"long"** format, with **one row = one measurement** (hour, station, pollutant, value).

## 2. Read one file: headers on one side, data on the other

The 6 header rows are read separately to build a small **metadata** table: one row per column of the file.

In [3]:
header = pd.read_csv(example, header=None, nrows=HEADER_ROWS, dtype=str)

meta = pd.DataFrame({
    "station_code":   header.iloc[2, 1:].str.strip().values,   # row 3
    "station_name":   header.iloc[1, 1:].str.strip().values,   # row 2
    "pollutant_name": header.iloc[3, 1:].str.strip().values,   # row 4
    "pollutant_code": header.i
    loc[4, 1:].str.strip().values,   # row 5
    "unit":           header.iloc[5, 1:].str.strip().values,   # row 6
})
meta.head(8)

,station_code,station_name,pollutant_name,pollutant_code,unit
0,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3
1,RUR_N,Zone rurale Nord - Saint-Martin-du-Tertre,"PM 2,5 particules",PM25,microg/m3
2,RUR-SE,Zone rurale Sud-Est - Forêt de Fontainebleau,"PM 2,5 particules",PM25,microg/m3
3,GON,GONESSE,"PM 2,5 particules",PM25,microg/m3
4,RAMBO,Rambouillet,"PM 2,5 particules",PM25,microg/m3
5,BP_EST,Boulevard Périphérique Est,"PM 2,5 particules",PM25,microg/m3
6,AUT,Boulevard Péripherique Auteuil,"PM 2,5 particules",PM25,microg/m3
7,BOB,BOBIGNY,"PM 2,5 particules",PM25,microg/m3


Then the data are read **without a header**, skipping the first 6 rows. The columns are numbered by their **position** (0, 1, 2…) and not by the station code, because the same code can appear twice. This is the case of `RD934` in the PM2.5 files: one full column, one empty column.

In [4]:
data = pd.read_csv(example, header=None, skiprows=HEADER_ROWS)
data.columns = ["timestamp"] + list(range(len(meta)))

print("Duplicated codes in this file:", meta["station_code"][meta["station_code"].duplicated()].tolist())
data.head(3)


Duplicated codes in this file: []


,timestamp,0,1,2,3,4,5,6,7,8,9,10,11,12
0,2018-01-01 01:00:00Z,7.1,7.1,7.9,6.4,3.0,8.9,NaN,6.1,6.3,10.7,9.3,9.2,10.0
1,2018-01-01 02:00:00Z,6.5,7.7,6.9,2.8,5.4,8.2,NaN,5.8,4.6,8.2,15.5,5.9,7.7
2,2018-01-01 03:00:00Z,8.2,5.5,3.7,2.6,6.0,6.1,NaN,4.0,2.4,8.0,7.7,4.8,4.0


## 3. From wide to long format (`melt`)

`melt` "unfolds" the table: each cell (hour, column) becomes a row. Empty cells are hours without a measurement (breakdown, maintenance): they are removed. Each row then gets the information of its column (station, pollutant, unit) thanks to the `meta` table.

In [5]:
long = data.melt(id_vars="timestamp", var_name="column_number", value_name="value")
print("After melt:", len(long), "rows")

long = long.dropna(subset=["value"])
print("After removing empty cells:", len(long), "rows")

long = long.merge(meta, left_on="column_number", right_index=True).drop(columns="column_number")
long.head()


After melt: 113880 rows
After removing empty cells: 109605 rows


,timestamp,value,station_code,station_name,pollutant_name,pollutant_code,unit
0,2018-01-01 01:00:00Z,7.1,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3
1,2018-01-01 02:00:00Z,6.5,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3
2,2018-01-01 03:00:00Z,8.2,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3
3,2018-01-01 04:00:00Z,9.6,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3
4,2018-01-01 05:00:00Z,8.9,RN6,Route Nationale 6 - MELUN,"PM 2,5 particules",PM25,microg/m3


## 4. Apply the same method to every file

Steps 2 and 3 are grouped into a **function**, applied to each file. We add a **safeguard**: if the 7th row of a file is not a date, its format is different and we stop with a clear message, rather than silently producing wrong data.

In [3]:
def read_airparif_file(path):
    """Read an Airparif export and return (measurements in long format, metadata)."""
    # Safeguard on the format
    with open(path, encoding="utf-8-sig") as f:
        lines = [f.readline() for _ in range(HEADER_ROWS + 1)]
    if not re.match(r"\d{4}-\d{2}-\d{2}", lines[-1]):
        raise ValueError(f"{path.name}: unexpected format, row 7 is not a date")

    # Headers -> metadata
    header = pd.read_csv(path, header=None, nrows=HEADER_ROWS, dtype=str)
    meta = pd.DataFrame({
        "station_code":   header.iloc[2, 1:].str.strip().values,
        "station_name":   header.iloc[1, 1:].str.strip().values,
        "pollutant_name": header.iloc[3, 1:].str.strip().values,
        "pollutant_code": header.iloc[4, 1:].str.strip().values,
        "unit":           header.iloc[5, 1:].str.strip().values,
    })

    # Data -> long format
    data = pd.read_csv(path, header=None, skiprows=HEADER_ROWS)
    data.columns = ["timestamp"] + list(range(len(meta)))
    long = data.melt(id_vars="timestamp", var_name="column_number", value_name="value")
    long = long.dropna(subset=["value"])
    long = long.merge(meta, left_on="column_number", right_index=True).drop(columns="column_number")
    return long, meta


pieces, metas = [], []
for path in files:
    long, meta = read_airparif_file(path)
    # Stations present in the header but without any measurement over the year
    empty = sorted(set(meta["station_code"]) - set(long["station_code"]))
    print(f"{path.name:<16} {len(long):>8} measurements" + (f"   no data: {empty}" if empty else ""))
    pieces.append(long)
    metas.append(meta)

measurements = pd.concat(pieces, ignore_index=True)
all_meta = pd.concat(metas, ignore_index=True)
print("\nTotal:", len(measurements), "hourly measurements")

2018 CO.csv         38142 measurements
2018 NO2.csv       334783 measurements
2018 NOX.csv       318140 measurements   no data: ['BAGN', 'PA01H']
2018 O3.csv        206279 measurements
2018 PM10.csv      191280 measurements
2018 PM25.csv      109605 measurements
2019 CO.csv         32550 measurements   no data: ['BASCH']
2019 NO2.csv       328389 measurements   no data: ['BASCH']
2019 NOX.csv       311806 measurements   no data: ['BAGN', 'BASCH']
2019 O3.csv        204885 measurements
2019 PM10.csv      182736 measurements   no data: ['BASCH']
2019 PM25.csv      107995 measurements
2020 CO.csv         41761 measurements
2020 NO2.csv       333419 measurements
2020 NOX.csv       316402 measurements   no data: ['BAGN']
2020 O3.csv        203313 measurements
2020 PM10.csv      197871 measurements
2020 PM25.csv      108468 measurements
2021 CO.csv          8826 measurements
2021 NOX.csv       312313 measurements   no data: ['BAGN']
2021 O3.csv        203220 measurements
2021 PM10.csv      1

## 5. Convert dates and values

**Dates.** The first timestamp of each file is `01:00Z`, not `00:00Z`: the timestamp therefore marks the **end** of the measured hour (`01:00Z` = measurement from 00:00 to 01:00 UTC). We subtract 1 hour to get the **start** of the hour, then convert from UTC to **Paris time**. A "day" then matches the day lived by Parisians, daylight-saving changes included.

*This reading of the timestamp is a reasonable assumption, to be confirmed in the Airparif documentation.*

**Values** are converted to numbers. Any unexpected text becomes `NaN` (missing value) instead of crashing the code.

In [4]:
utc = pd.to_datetime(measurements["timestamp"], utc=True) - pd.Timedelta(hours=1)
measurements["timestamp"] = utc.dt.tz_convert(TIMEZONE).dt.tz_localize(None)

measurements["value"] = pd.to_numeric(measurements["value"], errors="coerce")
print("Non-numeric values:", measurements["value"].isna().sum())
measurements = measurements.dropna(subset=["value"])

print("Period covered:", measurements["timestamp"].min(), "->", measurements["timestamp"].max())

Non-numeric values: 0
Period covered: 2018-01-01 01:00:00 -> 2026-09-21 22:00:00


## 6. Fix negative values

A concentration cannot be negative. We first look at **how many** there are and **how negative** they are, before deciding what to do.

In [6]:
negatives = measurements["value"] < 0
print(f"Negative value
s: {negatives.sum()} out of {len(measurements)} ({negatives.mean():.3%})")
print("Minimum observed:", measurements["value"].min())
print("\nBreakdown per pollutant:")
print(measurements.loc[negatives, "pollutant_code"].value_counts().to_string())

SyntaxError: unterminated f-string literal (detected at line 2) (585497917.py, line 2)

Negative values are **very rare** (a few hundredths of a percent) and **very close to zero** (at worst about −3 µg/m³). This is not a typing error but the **noise of the measuring device** when the air is very clean.

**Decision: they are set to 0.**
- We do not delete them: that would remove hours of clean air and artificially raise the means.
- We do not keep them as they are: they are physically impossible.

In [5]:
measurements.loc[negatives, "value"] = 0.0
print("Minimum after the fix:", measurements["value"].min())

NameError: name 'negatives' is not defined

## 7. Remove duplicates

Each yearly file also contains the **first hour of the next year** (for example `2019-01-01 00:00Z` at the end of `2018_PM25.csv`), which is also at the start of the 2019 file. The same measurement is therefore present twice.

The key of a measurement is the triplet **(hour, station, pollutant)**: it must be unique.

In [ ]:
key = ["timestamp", "station_code", "pollutant_code"]
print("Duplicates on the key:", measurements.duplicated(subset=key).sum())

measurements = measurements.drop_duplicates(subset=key)
print("After removal:", measurements.duplicated(subset=key).sum())

## 8. Build the dimension tables

**Stations.** The file does not give the station **type**, although it is essential: a **traffic** station (next to a road) is not comparable to an **urban background** station (which measures ambient air). We infer it from the name:
- a road name (boulevard, avenue, RN20…) means a **traffic** station;
- "Zone rurale" means a **rural** station;
- all other names are **urban background** stations.

`\b` means "start of a word". Without it, "Champigny-sur-Ma**rn**e" would be taken for a national road ("RN"). This error was spotted during the checks.

*This is a heuristic, to compare with the official list of Airparif stations.*

In [ ]:
def station_type(name):
    n = name.lower()
    if "rural" in n or "rambouillet" in n:
        return "rural"
    if "tour eiffel" in n:
        return "observation"
    if re.search(r"\b(boulevard|avenue|place|rue|quai|autoroute|route|rn\s?\d|rd\s?\d)", n):
        return "traffic"
    return "urban_background"

stations = (all_meta[["station_code", "station_name"]]
            .drop_duplicates("station_code")
            .sort_values("station_code")
            .reset_index(drop=True))
stations["station_type"] = stations["station_name"].map(station_type)
# Keep only the stations with at least one measurement
stations = stations[stations["station_code"].isin(measurements["station_code"])].reset_index(drop=True)

print(stations["station_type"].value_counts().to_string())
stations[stations["station_name"].str.contains("PARIS|Paris")]

In [ ]:
pollutants = (all_meta[["pollutant_code", "pollutant_name", "unit"]]
              .drop_duplicates("pollutant_code")
              .reset_index(drop=True))
pollutants

Only the useful columns are kept in the measurement table. The station name and the unit are in the dimension tables: repeating them on each of the 9 million rows would be a waste. This is the principle of **normalisation**, which we will find again in MySQL.

In [ ]:
measurements = (measurements[["timestamp", "station_code", "pollutant_code", "value"]]
                .sort_values(["pollutant_code", "station_code", "timestamp"])
                .reset_index(drop=True))
print(measurements.shape)
measurements.head()

## 9. Check the quality: coverage rate

For each station and each year, what **percentage of hours** has a PM2.5 measurement? This table is used to choose the target station and to spot the gaps.

In [ ]:
pm25 = measurements[measurements["pollutant_code"] == "PM25"].copy()
pm25["year"] = pm25["timestamp"].dt.year
hours_per_year = pm25.groupby("year")["timestamp"].nunique()   # possible hours in the year
coverage = (pm25.groupby(["station_code", "year"]).size()
                .unstack(fill_value=0)
                .div(hours_per_year, axis=1) * 100).round(0)

# Paris urban background stations only
paris_background = stations[(stations["station_type"] == "urban_background")
                            & stations["station_name"].str.contains("PARIS|Paris")]["station_code"]
coverage.loc[coverage.index.intersection(paris_background)]

**Reading:**
- **PA04C** ("Paris Centre", 4th arr.) measures until September 2019;
- **PA01H** ("Paris 1er Les Halles"), about 1 km away, takes over from October 2019;
- the two stations **never measured on the same day**.

They are joined into a single "central Paris" series. This is an assumption to mention in the report.

## 10. Compute the daily means (18-hours-out-of-24 rule)

A mean computed on only 3 hours does not represent a day. We apply the usual completeness rule: a daily mean is **valid only if at least 18 hours out of 24 (75 %)** are measured.

We also keep the **highest hourly value** of the day, useful for ozone, whose afternoon peak matters more than the mean.

In [ ]:
measurements["date"] = measurements["timestamp"].dt.normalize()

daily = (measurements.groupby(["date", "station_code", "pollutant_code"])["value"]
                     .agg(mean="mean", max="max", n_hours="count")
                     .reset_index())

incomplete = daily["n_hours"] < MIN_HOURS_PER_DAY
print(f"Days rejected (fewer than {MIN_HOURS_PER_DAY} h): {incomplete.sum()} out of {len(daily)} ({incomplete.mean():.1%})")

daily = daily[~incomplete].reset_index(drop=True)
daily[["mean", "max"]] = daily[["mean", "max"]].round(2)
measurements = measurements.drop(columns="date")
daily.head()

### Visual check: PM2.5 in central Paris

A chart shows at a glance what the numbers hide: gaps, outliers, a break when the station changes.

In [ ]:
centre = daily[(daily["pollutant_code"] == "PM25")
               & daily["station_code"].isin(["PA04C", "PA01H"])]
centre = centre.set_index("date")["mean"].asfreq("D")   # missing days -> visible gaps
monthly = centre.resample("MS").mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(centre.index, centre.values, color="#2a78d6", alpha=0.25, linewidth=1, label="Daily mean")
ax.plot(monthly.index, monthly.values, color="#2a78d6", linewidth=2, label="Monthly mean")
ax.axhline(15, color="#52514e", linestyle="--", linewidth=1)
ax.text(centre.index.max(), 16, "WHO guideline 15 µg/m³", ha="right", va="bottom", color="#52514e", fontsize=9)
ax.axvline(pd.Timestamp("2019-09-27"), color="#52514e", linewidth=0.8, linestyle=":")
ax.text(pd.Timestamp("2019-10-15"), ax.get_ylim()[1] * 0.92, "PA04C → PA01H", color="#52514e", fontsize=9)
ax.set_title("PM2.5 in central Paris, 2018-2026", loc="left", fontsize=12)
ax.set_ylabel("µg/m³")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e6e6e3", linewidth=0.8)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

**Reading:**
- the winter and early-spring peaks come back every year;
- the average level decreases slightly over the years;
- there is no visible break when the station changes, at the end of 2019.

This last point makes joining the two stations credible, without proving it.

## 11. Final checks and saving

`assert` statements automatically check the rules we have just applied. If one of them is false, the notebook stops: we never save data that do not follow our own rules.

In [ ]:
assert (measurements["value"] >= 0).all(), "negative value left"
assert not measurements.duplicated(subset=["timestamp", "station_code", "pollutant_code"]).any(), "duplicate left"
assert measurements["station_code"].isin(stations["station_code"]).all(), "unknown station"
assert measurements["pollutant_code"].isin(pollutants["pollutant_code"]).all(), "unknown pollutant"
assert (daily["n_hours"] >= MIN_HOURS_PER_DAY).all(), "incomplete day"
assert stations["station_code"].is_unique and pollutants["pollutant_code"].is_unique
print("All checks passed.")

stations.to_csv(PROCESSED_FOLDER / "stations.csv", index=False)
pollutants.to_csv(PROCESSED_FOLDER / "pollutants.csv", index=False)
measurements.to_csv(PROCESSED_FOLDER / "hourly_measurements.csv", index=False)
daily.to_csv(PROCESSED_FOLDER / "daily_measurements.csv", index=False)

pd.DataFrame({
    "file": ["stations.csv", "pollutants.csv", "hourly_measurements.csv", "daily_measurements.csv"],
    "rows": [len(stations), len(pollutants), len(measurements), len(daily)],
})

## Summary of the cleaning choices (for the report)

| Issue found | Decision | Justification |
|---|---|---|
| Wide format (one column per station) | Long format with `melt` | One row = one measurement, suited to a relational database |
| Duplicated station code (RD934) | Columns identified by their position | Avoids mixing up the two columns |
| UTC hours, end-of-hour timestamp | −1 h, then conversion to Paris time | Days aligned with Parisian life |
| Negative values (≈ 0.03 %, at worst −3 µg/m³) | Set to 0 | Device noise; deleting them would bias the means upwards |
| First hour of the next year duplicated | Duplicates removed on (hour, station, pollutant) | Unique key of a measurement |
| Stations without any measurement | Removed from the station table | No useful information |
| Station type missing | Inferred from the name (whole words) | Needed to compare background and traffic; to be checked |
| Incomplete days | Mean valid if at least 18 h out of 24 | Usual 75 % completeness rule |
| Station change in 2019 (PA04C → PA01H) | Joined into one "central Paris" series | Same type, about 1 km apart; assumption reported |

*This notebook does the same processing as the script `src/collect_flat_files.py`. The script re-runs everything automatically; the notebook explains and shows each step.*